2nd project Planning agent    ( Building a agent, this agent planes, researches , and white blogs automatically using langgraph , agentic ai project)

A planning agent is an AI agent that does not immediately jump to answering or acting - instead , it first creates a structured plan of what needs to be 
done , and then executes that plan step-by step

- planning agents are usefull when task are :
     - long or multi step

     - require structure(blogs, projects, apps)

phase 1 - plan
    break the task into clear steps or sub-tasks

phase 2 - execute
     complete each step systematically often checking progress



Graph of the agent "

start-> router(conditional edge router- research - orchestrator or  router - orchestrator )-> research -> orchestrator-> worker -> reducer - > end 

flow :
Start= in the start node the user will give us the topic or we will get a topic 

router = the purpose of the router node is it tells us if for the given topic we need to research on internet or not  if the provided topic info is not there with our llm that we go through this route router -> research -> orchestrator else router -> orchestrator if we have enough info

router node = the work of the router node is too tell us for the specific topic if we need to go on the internet and do research or not  thats it 


research = takes ur topic and break down it  in some research pointers  if u are going to research it so what will be the searches for our given topic, it plan the searches which our agent gonna do btw to conduct it searches we will use **Tavily= it is a search engine for llms**  so from tavily we will bring info from the tavily and pass this to the 

orchestrator=**orchestrator node and node is our planner node the work of this nodee to plan  for the blog plane in the sense what all section will be there in the blog based on llm knowledge and the research data on the basis of this out plannenr node will decide plane means decide the section of the blog and also decides how many words will be there in a section and if every section need research or not or citations need of code or not**

so at this stage we have our plan and the necessary info after this their comes a important step i.e worker node

worker node = worker node is that node which will write blog for us for this case we have 9 sectionsfor the blog so there will be 9 workers will work here u can see the 1 worker node but depending on your plan or the section that many workers will be there i.e worker 1 for 1st section worker 2 for 2nd section like this  and this all workers will do their work parallely so after some time all youe workers will finish their work and give u  after that reducers node will come 

reducer node = this node has 2 works it will take the sections and stich them all the section are stiched and the second task is to send this combined blog to the llm and ask it in this full blog where do u think we need to attach images and this node will generate images and attach the images where it is needed
2 task :
stich the sections
ask the llm where the image is needed and gen those image and add them


This is the architecture of the chatbot or agent 

We will develop it in 4 stages 

step 1] we will build a basic blog writing  agent this agent wont have the feature of researching  and also image  it will gen simple textual blog  

for this  basic blog the worflow will be 
start -> orchestrator[planner] -> worker -> reducer-> end 

using orchestrator because we are ot using anyinternet searching tool no thr route node is a conditional node so because here it is not there wo we are not using router   

step 2] we will add research feature in this bot i.e it can go on internet and bring the relevant information

step3] add the feature to add images to the blog 

step 4]  convert it into gui



**features**
can draw images whereever needed 

its has a research features it can go on the internet and do the research as like a research agent 

and in the plan section u can see which node is been executed u can see the progress it will show the section and how many words we need for that section and if every section needs research oor need citations , requires code or not

in the evidence tab we can see the citations[evidence]if u give any topic that needs research means top ai news in this week than the llm will do the web search and thenin the tab he will give the links  it will show the links citations

there is a log tab where we can see the decision making of the agent 

and the images tab where we can see the images that have been generated 

this is the complete blog writing agent

Reducer functions help determine how values in the state should be updated when changes are made. Without a reducer, any update will simply overwrite the previous value. But with reducers, we can customize how each part of the state behaves when updated.

step 1] we will build a basic blog writing  agent thois agent wont have the feature of researching  and also image  it will gen simple textual blog  

for this  basic blog the workflow will be 
start -> orchestrator[planner] -> worker -> reducer-> end 

so in this we have orchestrator that is a planner and the worker node that are the worker which will gen blogs  

so in the start the user wil give some topic for ex self attention this topic is forwaded to the orchestrator(planner) what it will do it will see the topic and build a plan around it we will have  plan object which will be a pydantic object and the scchema of it means plane it this:
plan( this plan object will have 2 things 1st is blog title and set of tasks[task 1 , task 2 like this] this tasks are the task objects which are also a pydantic object see the below for the task  )
plan:
blog_titl(str):
    tasks(list(Task))


Task(each task)(each task will have the following things)
id(int), task id
title(str),
brief(str, description : what to cover)

each task object will contain info about one section

this are the pydantic object it is a schema becasue we want output in a specific format,and for each task we have id ,title and breif of that particular section


so as soon as topic reaches the orchestrator it will gen a plan and plan will have one blog title and list of task object(plan[blog title,[list of tasksobject]])  each task object will contain info about  one section of the blog so if we have 5 sections for out blog so for each section corresponding one task will form and each task object it will have taks id that sections title and brief( what we will put in this section that info in brinf u can see in the task function)

as soon as the it reaches the orchestrator it will gen plan in plan we give the blog title and task objects each task object will contain the id id of 1 section title of that section and descroption what should be written in this section if at planning orchester will decide there should be 5 section so their will be  task task=section  and this plan and 

**so till now we have plan we wil send this plane to worker node **but her a mechanism we will use called**
**orchestrator worker flow in this we fanout  or trigger that much worker need for the task if 5 task are ther 5 worker will be fanout or trigerred   how much task u have/how much u want so if u have 5 task to do than automatically 5 works will run and orchestrator will provide one task to each worker and in the task the description is how to write that section and each worker node start to do their work for that task ot section and each worker node has its own llm and paralleli all this section are executed once it is executed we will stich them means join them and send to the reducer this reducer have 2 task th stich the section and make a blog and create a mardown file and this markddown file will have the final blog**

 

For the 2nd step we  are making changes in the system propmts in the orchestrator(planner)and the reducer(which is making the md file) the propmt which we gave is too basic earliear the prompt was 

def orchestrator(state:State) -> dict:
    plan= llm.with_structured_output(Plan).invoke( #the output of this llm should always be a plan object , the output  will be a pydantic object.we are calling the llm and telling him the plan object  .The output should always be a plan object
        [
            SystemMessage(content=("create a blog plan with 5-7 sections on the following topic")),
            HumanMessage(content=f"Topic: {state['topic']}")
        ]
    )
 
    return {'plan':plan}


also made changes in the task object added more details 


def worker(payload: dict) -> dict: (this worker write section each worker will write one section )

    # payload contains what we sent
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke( #  this worker will write and store section in the md file but like this many worker will do its work each worker will do its work and store in the md file wirker will read the system message and human one in human we will provide every info
        [
            SystemMessage(content="Write one clean Markdown section."),
            HumanMessage(
                content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Return only the section content in Markdown."
                )
            ),
        ]
    ).content.strip()

to make our agent better we have to give more elaberate systems prompt because this prompts are too basic we will do 2-3 chages 

change  1
We have the plan object(its a pydantic schema) which orchestrator make  we will add 1-2 details in this  previously we had this 

class Plan(BaseModel):
    blog_title: str

    task: list[Task]

now we will add audiance and tone

in the task we have multiple chabges added more detail , and in the orchestrator node which is our planner node we have given a detailed system prompt to it
and same for the worker node 


#step 2


**Now we will add  research dfeature in our blog writing agent if we gave the topic and te topic is of recent means development in ai so our llm can go on the web and search and bring the answer this is a very important feature because if u see any research assistant they have this feature because they go to the internet and research**

But the code of it is little diffcultwill make new file and understand the difference 

**The plan to make this system**

The plan is simple when we get out topic in the start node and this router(router node) it is a llm based node  and this router has a simple work it has to decide seeing the topic if the topic needs research or not research means the llm will go to thr internet and get the relevant info of that topic if the topic is self attenttion we do not need the research because the self attention is already there in the parametric knowledge and there is not a change in the concept  if ur topic is top ai news then u need togo na search because the model is tuned are completed its training in 2024 because of this the model has no knowledge of the ai news in 2026

if the router is identifies that we have to do internet search ,not only he identifies that we have to do internet search but also gives /recommends
the search queries for ex if the q is "evolution of chatgpt from 2022 to 2026 for this particular blog/topic we have to do internet search and he willalso suggest us the search queries which will help writing the blog  when we see which node is executed in the plan section we can see it and in the plan section when the router identify the research and below we can see the queries it recommend now with the help of router we get we ned to do research and also and also queries the router is giving 2 output here now this 2 will go to the research node 

Research node will use the tavily(search engine for llms) the llm willtake the queroes one by one and send it to the tavily and this tavily will go and give u info againts the queriesgiven by the router and store it in the state


Now we will go to the orchestrator(our planner node) the tell him we have this blog topic and we wanna generate plan and tells us what sections should be there and yes we have the info from the tavily from the interbet we will tell the orchestrator to consider this info little bit on the basis on the topic and the provided info  on the basis of it he will gen plan

The plan we think and gen plan and he decided to make 5 section Now this **5 sections will go to 5 different workers nodes and each worker node independently do work for each section means this section and for writing particular section if the worker needs the info from the internet we will provide him the info stored in state this infocame from tavily via research node  like this all the section will be prepared 

and after this our reducer node will stich this section and we willget out blog 

router(decides if need research and gives us q related to the topic )-> researchnode(usese tavily and give the info aganist the queries and stores this info in state)->orchestrator(planner, tell him to gen plane and give him the info by seeing this he can gen plan[ jo section create ho rahe hai,and for creating if he needs the internet info he can use it the info is in state ])-> worker[the worker wil write sections and if he needs info he can use the info stored in the state ]

**Now the workflow of the graph**

**start->router(condition edge)->research->orchestator(this flow if need researchif not->router->orchestrator->worker(after router there is no conditional edge, this worker writes the section independantly)->reducer(merges sections and in the future decides wheather to add images or not)->end**


schemas to implement this research flow / updated schemas:

  
routertdecision:
needs_research:bool
mode:Literal['closed_book','hybrd','open_booj']
queries;[List(str)]

Now we have a router and router decides if we need to research or not on the bases of the topic . when this topic goes to the router note it gives us the router decision as an output which is the pydantic schema and from this we are getting 3 output
1] needs_research= true or false[boolean]

2]mode =its a key and we are getting its value it has 3 possible velue closedbook,hybrid,open book

- closed_book= means the topic like self attention that doesnt need to go and research

- hybrid =for this topic u can give from the parametric knowledge but also needs to search for ex open source models u can tell what is open source knowleedge from the parametric knowledge but the q is like " what is the open source models and what are the latest open source then u need to go on the internet and search

- open_book = top ai news of the week for this we need to go and search in the internet 


so our router is saying us wheather we need to do research or not  and in the mode he is saying in which category it is comming  for closed book u dont need to go and search and for the rest 2 u have to go and research and for queries if the router tells that the research is needed then it generates the queries base on the topic given to router if mode is hybrid or open 

assume we are comming to t he yes part means we need to research as u know the flow start->router-> research , now we have to conduct research we have the queries generated by the router  so now for each query we will do a tavily search

To use tavily just lofgin in there web anad u will get the api just paste that api in the env file and in code just like duck duck search we have langhain_community.tools import  TavilySearchResults and in this u will tell means there is a parameter u will tell that against the search results how many  search results u want  and invoke it buy giving it a query and by running loop u can fetch all this . of every q u will do a tavily search and we will get our results and u standardize every results i.e You covert it into standardize format and this format we call **Evidence item,which is a pydantic schema**

so for every tavily search result we are storing its title ,url  , at which day it is publised , source and snippet snippet means the content 

EvidenceItem:
title:str title or the query 
url:str url of the info
publised_at:optional[str] which date it was publised means the info 
snippet:optional[str]
source:optional[str]

so what our researcher is doin when he gets 5 query fromm the router node  for each query it is going and doin a tavily search and for every tavily search he is getting 2-3 ans depends what u have set in the tavily means the max_results assume we have 5 q and this max result is set 2 then for the 5 q we will get total of 10 ans/results and every result object we have converted it into a evidenceitem which has title , url,publised data asn the snippet[content] ad the results was 10 so out evidenceitem is also 10 so we are converting into a **Evidencepack[it is collection of evidenceitem object ]**
so we are getting 1 evidencepack object and evidencepack is the collection of multiple evidenceitem objects **and we will store this evidencepack in the state**

evidencepack:
evidence:List[evidenceitem]

Now we are going to the orchestor and we will tell him to generate the plan with the help of the plan object and what the plan should have blogtitle, audience,tone.blog_kind task and task means section's details  and each task is this task object mwans the pydantic task each task has it id,title,bullets,target_words,tags,requires_research,requirescitations,requires_code all this  this is present in the task pydatic schema remember task means sections details what each section shoud have each task willl have there own

orchestrator will make a plan and in the process f making plan the researcher has given him the evidencepack object  isse bhi ye idea lega or plan banayega 
and he will make the task and hhe will send each task to the workers 1 worker will have only 1 task like this 5 task 5 works task=sections and workers will write section independantly and when the worker needs the internet info it can go to the evidencepack and take the info and now we will have the sections now it will go to the reducer it will stich it join the diff section into a single and a blog 

In [ ]:
#tavily
from langchain_community.tools import TavilySearchResults
from dotenv import load_dotenv
tool=TavilySearchResults(max_results=3)
results=tool.invoke({'query':'chatgpt version releases and updates from 2022 to 2026'})
results



[{'url': 'https://wpexperts.io/blog/chatgpt-version-history',
  'content': '## Markets We Serve\n\n# ChatGPT Version History – Latest Releases & Future Scenario\n\nUsman Hayat\n\nUpdated onJuly 1, 2026\n\nChatGPT Version History\n\nBy the end of 2022, OpenAI had familiarized the world with ChatGPT. Since its launch, ChatGPT has shown no significant signs of slowing or regressing in introducing new features or maintaining worldwide user interest. Continue exploring the history of ChatGPT with a timeline of various versions and updates. This blog post will highlight the ChatGPT version history in detail. Besides, it will briefly shed light on its future.\n\n## What Do You Mean by ChatGPT?\n\nWhat Do You Mean by ChatGPT_ What Do You Mean by ChatGPT_ [...] Performance Update\n\nDecember 2022\n\nChatGPT launched a massive update that helped user experience, usability, and response time.\n\nChatGPT Launch\n\nNovember 2022\n\nChatGPT started its journey and introduced key advancements in NLP 

In [4]:
for r in results:
    print(r['content'])

## Markets We Serve

# ChatGPT Version History – Latest Releases & Future Scenario

Usman Hayat

Updated onJuly 1, 2026

ChatGPT Version History

By the end of 2022, OpenAI had familiarized the world with ChatGPT. Since its launch, ChatGPT has shown no significant signs of slowing or regressing in introducing new features or maintaining worldwide user interest. Continue exploring the history of ChatGPT with a timeline of various versions and updates. This blog post will highlight the ChatGPT version history in detail. Besides, it will briefly shed light on its future.

## What Do You Mean by ChatGPT?

What Do You Mean by ChatGPT_ What Do You Mean by ChatGPT_ [...] Performance Update

December 2022

ChatGPT launched a massive update that helped user experience, usability, and response time.

ChatGPT Launch

November 2022

ChatGPT started its journey and introduced key advancements in NLP (Natural Language Processing).

The above ChatGPT version list indicates that previous versions of C

**Step 3 adding image: blog writing agent with images**

SO for the image we only have to do changes in the reducer part .till now u know our reducer is oin 2 works the section written bu the workers the reducer merges that all the section which becomes the overall blog and write this blog in the file means the md file 

Now to add the image functionality our reducer willdo extra work

step1 = the reducerr willdo the merging part of the sections make a blog and write in the md file


**step2**=In step 2 the reducer will send this markdown  to the llm and ask that by seeing this blog what u think we should add images or not i.e in this textual content or blog in which places/part  we shoud add images the llm will read the blog and tell us where to add images not only our llm is telling us where to add images but also which type of image should we add so he will create a placeholder thw placeholderis created where the images should be added like [[image_1]] ,[[image_2]] like this  not only the llm will create the placeholder but also gives us the prompt how to generate this image for ex : the prompth will be for this particular image this should be the file name(for that particular image) and he will give us the prompth and that prompt we willgive to the other llm(gemini) and what should be the nature of that image

from the step 2 we get 2 things  =

- a markdown  with placeholders(for image)

- prompts to produce the images using other llm. for every placeholder we have the prompts
 

**step 3**= in step 3 u give this prompts to a image generation models like gemini . u give all your prompts to this model and generate the image once all the images is generated u save all this images in your projects folder in the image directory(folder) . and when u save it  you replace the placeholder with this filename  every image has its file name and when u save this new markdown including the image when we open this new file we can see the text with the image . give the prompt to geminn adn download the image and store it into the your project folder

**Your reducer do this 3 things 

- merge the section generated by the workers
-  send this markdown to the llm and ask if there is a need of the images our llm will tell our where there is need of the image in which part we need it and he gives us the placeholder and prompt(this prompth will include the file name of that specific image and prompt to gen image from the another llm like gemini and download that image to the folder)**


**code part**:
Now we have converted our reducer node into a subgraph . means the reducer is not a single node but converted it into a subgraph

subgraph = start->merge_content-> decide_image-> generate_and_place_images->end

reducer in itself is the collection of 3 nodes 

- merge_content(node)= merges the sections and gen the markdown(blog) this merge log ot markdown is sent to thenext node decide

- decide_image(node)= send this blog/mardown to the llm and ask whthere is a need of images if yes in which part the llm will gives us the prompt gen placehoders and corresponding to each placeholder he will create a filname we will save image in this file 

The output of this decide_images node will have 2 outputs:
- markdown for placeholders 

- for each placeholder we will have this prompt structuture 
 
to represent this in a structured manner we have made a pydantic object(schema) that the output should bee like this we want output like this 

GlobalimagePlan: # ts is the pydatic schema of decide_image 
md_with_placeholders:str ( the blog with placeholder in step 2 its a step 2 output)

images:list[imageSpec] list of the pydantic model imagespec



imageSpec(pydantic schema): -> the llm ans should contain all this in the decide image node
placeholder:str

flename :ste where we will save the image

alt:str

caption :

prompt :str

size:Literal[1024*1024,'1024*1536','1536*1024]

quality: Literal["low','medium,'high]

so the decide image node will have 2 outputs 
- mardown with placeholders

- list of the image spec(in global image plan u can see, for each image u will have the this imagespec object and in the global image u will send the list of it)


next node 
gen_and_place_images(node)= for each image (placeholder)  he will feth the pydantic schema (imagespec) and go to gemini and gen image and he will eeplace the placeholder with this generated image 


the important output for this graph is the decider_image one because it gives us the placeholder where the image should be there and the prompt means the image spec for that particular image . for each placeholder we have image spec object i.e the llm should give us the details whivh is there in the schema like this we want our output 

for the image generation u need gemini api key how we can get this see go to the google ai studio-> create api key create a new project 







# see remember whe the go from th research path and we go through research means opn book or hybrid we ned to research than the blog will contain citatitons means rhe worker will provide the like url of the source it considered 

6
7
8
9
10



we faced too main problem and the biggest problem token per limit

**so the problem occured witn this system is ain tpm(token per limit) for this in the decide image did changes means the node was getting full markdown because of this this token problem was comming to gave only first 6k tokens bur then also while don research the problem came because of workers every task needs one worler to writ 1task means one section and earch worker was taking 2.5 tokens so for 5-6 worker the tokens is 12k and the limit is 8k to reduce this reducing this task for 5-6 to 3-4 so now 3-4 workers will work write th section**

This is a different problem than before — read carefully: model TPM limit isn't the issue here, message size is. The error says Requested 13609 tokens in one single call — that's your full merged blog (all 5 sections combined) being sent into decide_images at once.

Why a model swap alone won't fix this one:
Every free-tier model we've seen tops out around 8,000–12,000 TPM. Your single request is 13,609 tokens — that's bigger than even the highest free-tier cap (12,000 on the now-deprecated llama-3.3-70b). No free model swap can absorb a single call that large; this isn't a parallel-burst problem like the worker node was.

The honest reality: to fix this without paying for Dev Tier, you need to reduce what's sent into decide_images — that's a genuinely necessary one-line change, not optional:


the solution 

image_plan = planner.invoke(
    [
        SystemMessage(content=DECIDE_IMAGES_SYSTEM),
        HumanMessage(
            content=(
                f"Blog kind: {plan.blog_kind}\n"
                f"Topic: {state['topic']}\n\n"
                "Insert placeholders + propose image prompts.\n\n"
                f"{merged_md[:6000]}"  # <-- truncate here
            )
        ),
    ]
)

This caps the blog content sent for image-planning at ~6,000 tokens instead of the full merged doc, bringing your total request comfortably under 8,000.

Why this is safe to do: decide_images only needs enough of the blog to understand its structure and propose 1-3 diagram ideas — it doesn't need every word of a 13K-token blog to do that job well.

This is the one unavoidable code change — no model swap gets around a single request that's larger than every available free TPM ceiling.

What decide_images currently does:

merged_md = state["merged_md"]  # this is your FULL blog — all 5 sections combined, ~13,600 tokens worth of text
...
HumanMessage(content=(... f"{merged_md}"))  # sends the ENTIRE thing to the LLM

So yes — right now, the whole blog (every section, full text) gets stuffed into one message and sent to the model. That's why the request is 13,609 tokens — it's literally your complete blog.

What [:6000] (slicing) means:
In Python, merged_md[:6000] means "take only the first 6,000 characters of this string, cut off the rest."

**The core problem was Groq's free-tier token-per-minute (TPM) limit, which sits at 8,000 tokens for every model I used (gpt-oss-120b, gpt-oss-20b, qwen/qwen3.6-27b), regardless of the model's size. This became a real constraint because my architecture fans out 5-6 workers in parallel to write blog sections simultaneously, and each worker call used roughly 2,600 tokens — meaning all workers combined needed close to 13,000-15,000 tokens landing in the same 60-second window, well over the 8,000 ceiling. The first fix was reducing planned sections from 8-9 down to 5-6, which brought the parallel burst closer to the limit, though not fully safe on every topic.

A second, separate TPM issue showed up in the decide_images node, where I was sending the entire merged blog (about 13,609 tokens) into a single LLM call to plan image placements — again exceeding 8,000 TPM in one request. I fixed this by slicing the input to the first 6,000 characters (merged_md[:6000]) before sending it to the model, since the image-planning step doesn't need to see the full blog text to propose a few relevant diagrams.

That slicing fix introduced a new bug, though: the generate_and_place_images function was using the LLM's rewritten, truncated version of the blog as the final saved output, instead of the original full text. This meant my final blog was silently cut off at whatever point the 6,000-character slice ended. I fixed this by pointing the final-write step back at the original state["merged_md"] instead of the LLM's truncated rewrite.

On the reliability side, I hit two separate "tool did not call correctly" errors from Groq's structured-output mechanism — once in the decide_images node (where I fixed it by swapping from the smaller gpt-oss-20b to the more reliable gpt-oss-120b), and once in the router node, which I fixed by setting temperature=0 across my main LLM to make its structured-output tool-calling more consistent. A related schema validation error occurred when the model returned an image size value ("1024x768") outside my allowed enum — I fixed this by explicitly listing the valid size options directly in the system prompt so the model couldn't invent new ones.

I also ran into an output-quality bug where qwen/qwen3.6-27b, being a reasoning model, was leaking its internal "thinking" text directly into the blog sections instead of just the final answer. Setting model_kwargs={"reasoning_format": "hidden"} resolved this by telling Groq to strip the reasoning trace and return only the clean output.

The most confusing bug, though, involved checkpointing. After adding SQLite-based checkpointing (so the graph's state could persist and resume, which I need for my Streamlit frontend), I started getting a blog with only a title and no section content — with no error thrown at all. After testing with the checkpointer removed and confirming the graph worked perfectly without it, I traced the real cause: I had been reusing the same thread_id across multiple test runs, some of which had crashed mid-execution. LangGraph was resuming from that broken, partial checkpoint state instead of running the graph fresh. The fix was twofold — always use a new thread_id for every test run, and delete the corrupted database file whenever a run crashed midway.

Finally, I ran into a token-per-day (TPD) limit, separate from TPM, after enough cumulative testing used up the 200,000 daily token budget on qwen/qwen3.6-27b — a reminder that Groq enforces both a per-minute and a per-day ceiling, and heavy iterative testing during development can exhaust the daily one even when each individual call is well within the per-minute limit.

Yo remember to swithch the qwen model back in the worker node 





You're right to push back — yes, both levers increase tokens, no way around that. Let me be precise about how much each one increases risk, since they're not equal.

Lever 1 — raising target_words (250-400 instead of 120-320):

Mainly increases output tokens per worker (the model writes more)
Roughly: 320 words ≈ 430 tokens → 400 words ≈ 540 tokens (output side only goes up ~110 tokens per worker)
With 5-6 workers: extra ~550-660 tokens total across all workers combined

Lever 2 — raising section count (5-6 → 7-8):

Adds entire extra workers, each with full input (system prompt + evidence + bullets, ~500-800 tokens) PLUS full output (~400-500 tokens)
Each additional worker costs ~900-1,300 tokens, not just ~100-150

The real comparison: 1 extra section costs roughly 8-10x more tokens than just raising the word target on existing sections. That's why I called section count the "riskier lever" — it's not that tokens don't increase either way, it's that the increase per unit of "more content" is much smaller with word-count than with worker-count.

Concrete numbers, staying at 5-6 workers, raising target_words to 400:
5-6 workers × ~950 tokens (with the higher output) ≈ 4,750-5,700 tokens — still safely under 8,000.

If you push to 7 workers at the original 320 words:
7 × ~930 tokens ≈ 6,510 tokens — also technically fits, but with less safety margin, and any topic needing more evidence/citations pushes it closer to the edge.

Bottom line: yes, both increase tokens — I'm not saying otherwise. I'm saying raising target_words gives you more content per token spent than raising section count does. Your call on how much margin you want to keep.

merge section node


seen[task_id] = md — this way. Let me clear the confusion.

Reading it left to right: "seen at key task_id gets assigned the value md" — meaning md is being stored into the dictionary seen, using task_id as the key.

Not the reverse (md = seen[task_id]) — that would mean "look up whatever's already stored at seen[task_id], and put it into md" — that's reading from the dictionary, which is backwards from what we want here (we're writing into it).

Simple rule: in Python, x[key] = value always means "store value at key in x" — the dictionary/variable being written to is always on the left of =.

So seen[task_id] = md = "save this section's text (md) into the dictionary, filed under this task's ID (task_id)."





ay state["sections"] looks like this (with the duplicate bug — 3 tasks, but task 1 got written twice):

python
state["sections"] = [
    (1, "## Intro section text..."),
    (2, "## Second section text..."),
    (1, "## Intro section text... (duplicate)"),
    (3, "## Third section text...")
]

Step 1: seen = {}
Creates an empty dictionary — this will store one section per task_id, no duplicates allowed (dictionaries can't have two entries with the same key).

Step 2: for task_id, md in state["sections"]:
This loops through the list one tuple at a time. Each tuple is (task_id, md) — so on each pass, Python unpacks it: task_id gets the number, md gets the markdown text.

Step 3: if task_id not in seen:
Checks: "have I already stored something for this task_id?"

Step 4: seen[task_id] = md
Only runs if the check above is True — stores this section's text under its task_id key.

Walking through the example:

Loop iteration	task_id	Already in seen?	Action
1	1	No	Store it → seen[1] = "Intro..."
2	2	No	Store it → seen[2] = "Second..."
3	1	Yes (already there)	Skip — duplicate ignored
4	3	No	Store it → seen[3] = "Third..."

End result: seen = {1: "Intro...", 2: "Second...", 3: "Third..."} — exactly 3 entries, the duplicate silently dropped because the if task_id not in seen check blocked it from overwriting.

till now what problems we faced give that and also the solution we used

Blog Writing Agent — Problems & Solutions Log
1. TPM (tokens-per-minute) rate limit — parallel workers
Groq caps each model at ~8,000 TPM. With 5-6 workers firing in parallel (Send() fan-out), combined token usage (~13,000+) exceeded the limit.
Fix: Added try/except retry with a 20-22 second wait on the worker's LLM call; later confirmed the model itself (qwen/qwen3.6-27b) was capped at exactly 8,000 TPM via testing. 

**Tokens per minute (TPM) means the combined total of both sent (input) and received (output) tokens that your application can process within a single rolling 60-second window.**

2. TPM exceeded — single large call in decide_images
Sending the full merged blog (~13,600 tokens) into one call for image planning exceeded the same 8,000 TPM limit.
Fix: Sliced input to merged_md[:6000] before sending to the image-planning LLM.

3. Output truncated — using the LLM's rewritten text as final doc
After slicing for image planning, the final blog was built from the LLM's truncated rewrite (md_with_placeholders) instead of the original full text, silently cutting the blog short.
Fix: Changed the final-write step to use state["merged_md"] (the original, untruncated blog) instead.

4. "Tool did not call a tool" — Groq structured-output failures
Intermittent Groq reliability issue where with_structured_output() failed to return valid tool calls, hit in both router and decide_images.
Fix: Set temperature=0 on the affected LLMs to reduce randomness in tool-calling; added retry logic as a backup.

5. Schema validation failure — invalid size enum value
Image model returned "1024x768", not in the allowed Literal["1024x1024", "1024x1536", "1536x1024"].
Fix: Explicitly listed the exact allowed values in the system prompt.

6. Reasoning text leaking into blog content
qwen/qwen3.6-27b (a reasoning model) was outputting its internal "thinking" process directly into blog sections.
Fix: Added model_kwargs={"reasoning_format": "hidden"} to strip reasoning from the output.

7. Empty blog output — reused thread_id with checkpointing
Reusing the same thread_id across test runs caused LangGraph to resume from broken/partial checkpoint state instead of running fresh, resulting in an empty final blog with no error.
Fix: Use a new thread_id for every run; delete the checkpoint DB after any crashed run.

8. TPD (tokens-per-day) exhaustion
Heavy iterative testing used up the 200,000 daily token budget on qwen/qwen3.6-27b, separate from the per-minute limit.
Fix: Waited for daily reset, or temporarily swapped to a different model with a separate token pool.

9. Duplicate sections — race condition with parallel checkpoint writes
SQLite checkpointing combined with parallel worker writes occasionally caused a worker task to be retried internally, duplicating sections (e.g., 5 tasks → 10 sections).
Fix: Added a dedupe step in merge_content — kept only the first section per task_id using a dictionary before merging.

10. Checkpoint serializer crash — package version mismatch
Installing langgraph-checkpoint-postgres (explored as an alternative) pulled in an incompatible langgraph-checkpoint version, breaking the SQLite checkpointer (AttributeError: 'JsonPlusSerializer' object has no attribute 'dumps').
Fix: Uninstalled the Postgres package, pinned langgraph-checkpoint-sqlite==2.0.2 (matching a known-working version from another project) using pip install --break-system-packages.

11. Images requested but never appearing — placeholder mismatch
After fixing #3 (using full merged_md instead of the LLM's rewrite), the [[IMAGE_1]] placeholders no longer existed in the text being edited, so image insertion silently found nothing to replace.
Fix: Redesigned generate_and_place_images to append all generated images at the end of the blog under an "Images" section instead of relying on inline placeholder replacement.

12. Wrong/unavailable image model (imagen-4.0-fast-generate-001)
Model returned a 404 — not found/supported for the API key's access tier.
Fix: Swapped to gemini-2.5-flash-image; still hit a limit: 0 quota error (account-level, not code-related).

13. Google Gemini image quota — account-level limit: 0
The Google API key had zero free-tier quota for image generation, regardless of model name.
Fix: Switched to Pollinations.ai, a free, no-API-key image generation service — swapped the image-generation function's internals while keeping the same function signature so no other code needed to change.

14. Worker silently returning empty content
After switching workers to qwen/qwen3.6-27b, every worker returned .content as an empty string with no exception thrown — a silent failure the retry logic couldn't catch (since no error was raised).
Fix: Swapped workers back to openai/gpt-oss-120b, which resolved it; confirmed via debug prints showing Section lengths: {1: 0, 2: 0, ...} before the fix and real lengths after.

15. Images generated but not placed in the correct section
Images were appended at the very bottom of the blog, disconnected from the section they illustrated.
Fix: Added a related_section field to ImageSpec and prefixed each image with a "Illustrates: [section name]" label, without attempting risky inline placeholder insertion.



the thired problem i faced was the llm was hallucination when he was not getting source and was telling wrong percentage and info when we go through the research path this is only in research path not when the topic can be answed from the parametric knowledge this is the issue i can tell in interview and to resolve this i tighten the system prompth it reduced 50-60 percent 


what about the blog writing is taking 5-6 mins

That's a real, honest limitation worth understanding — but not necessarily something to fix tonight. Here's the breakdown of why it takes that long, so you know what to say if asked.

Where the time actually goes:

Router call (~2-5 sec)
Research node — Tavily searches (3 queries × network calls) + evidence extraction LLM call (~10-20 sec)
Orchestrator call (~5-10 sec)
5-6 parallel workers — even though they run in parallel, each is generating 250-400 words of markdown, which at typical LLM output speed can take 15-30 sec each; if any hit a rate limit and retry (your 20-22 sec wait), that adds real time
Image planning call (~5-10 sec)
Image generation — Pollinations calls (can take 10-30 sec each, sometimes timing out at 30 sec) for up to 3 images, run sequentially in your current code
The biggest single contributor is likely images — 3 sequential Pollinations calls at up to 30 sec each (with your current timeout=60 if you bumped it) can alone eat 1-3 minutes, especially if one times out and you still wait the full timeout before falling back.

Honest, simple thing you can say in an interview about this:

"The full pipeline takes 5-6 minutes, dominated by the parallel worker writing stage and sequential image generation. Streaming progress to the frontend was important specifically because of this — so the user isn't staring at a blank screen wondering if it's frozen."

That's actually a good answer — it shows you noticed the latency and solved the UX problem (streaming) even though you didn't fully solve the speed problem. That's honest and reasonable.

If you genuinely want to speed it up (optional, not urgent): the biggest lever would be running image generation calls in parallel (using asyncio or threading) instead of sequential loops — but that's a real code change, not something to do at this hour.

My recommendation: leave it as-is, document the "why," and stop here. 5-6 minutes for a multi-agent research + writing + image pipeline is genuinely reasonable and expected — nobody would expect this to be instant.



**The research blogs that are good to put in git**
so what u think what should i do because there are blos where there is correct citations in research one


Synthesized practical guidance for research citation decisions
Synthesized practical guidance for research citation decisions
Good point to raise — and it's true, this matters for how you frame things. Let's be precise about what you actually have.

The honest pattern across your research blogs tonight:

Cloud pricing comparison — cleanest, most accurate (real, specific, verifiable numbers)
Vector database comparison — mostly good, some "N/A" placeholders (honest gaps, not fabrication)
Kubernetes (first run) — good, real citations, real product names
AI benchmarks, quantum computing, smartphones, diabetes — clear fabrication of specifics
So the real pattern isn't "always hallucinates" — it's topic-dependent, and you can actually characterize it precisely:

Grounding works well when:

The topic covers stable, well-documented, slow-changing facts (cloud pricing pages, established open-source products)
Evidence snippets likely contain the exact numbers being asked about (pricing pages list exact dollar amounts directly)
Grounding fails when:

The topic asks about bleeding-edge, rapidly-changing, or not-yet-real specifics (unreleased phone models, hypothetical 2026 AI models, medical trial data)
The model's training data has strong "priors" about what a plausible-sounding answer should look like, so it fills gaps confidently even when evidence doesn't support it
What I'd actually do: reframe this as a finding, not a flaw — and say it precisely, using your real examples.

For your README/interview answer, replace the vague "it sometimes hallucinates" with something more specific and credible:

"Grounding accuracy varies significantly by topic type — for stable, well-documented subjects like cloud pricing (verified against real per-hour rates) or established open-source tools, citations and specifics were reliably accurate. For fast-moving or speculative topics — like hypothetical next-gen product specs or recent AI benchmark comparisons — the model would still generate confident, specific-sounding numbers not fully supported by the retrieved evidence, despite explicit anti-fabrication instructions."

This is a much stronger, more credible statement than a blanket "it hallucinates." It shows you actually characterized the failure mode precisely, across multiple real test cases — which is a stronger signal of engineering rigor than either overclaiming ("it's fixed") or underclaiming ("it's broken").

Practical next step: nothing more to build tonight. Just update your README with this more precise framing instead of the general limitation note — that's a 5-minute edit, not new code.



And the main reason we send the llm temperature see if we set the temperature 1 or more than one than the llm tries to make the ans creative or use random words ti make the output more creative and when the temperature is between 0.3-0.5 means low than the llm choose the words which has most probability thi our blog writing we set it 0 because when the worker node was writing the section he was type of hallucinating or hee was not calling the tool properly so we set it to 0 and then the problem is solved he was giving only that answer which he reseached from the web using only that words didnt hallucinated 

interview q 

see if the interviewer ask we can do this in the single llm call the ans u should give is yes for every short blog we cant do but that llm called cannot go on internet ad search for the info he cannot bring the citations he cannot bring the info if the topic is very recent topic and also remember thsi blog is written in the md file ad it is saved in our system folder so normal llm cannot do all this u can tell this 
 


and always remember u faced the difficulty of token per minute problem for that we reduce the no of sention total words and and system prompt and also re did try excpet for the worker node so that it waaits 20 sec and try again 


when the interview what alse is the tradeoff of this project or what will u imporve u can tell that i will imporve at the writing part because we are calling/making too many llm call so to reduce that we can combine small works into one llm for exaplin merging decide imgae llm inn th merge node make the workflow less complex 


and when the interviewer ask  why u used subgraph instead of normal graph so u can tell we can do this witout a subgaph because my reducee was doin three things previosly he was doin 1 thing thet is merge the content but now he is doin 3 thingns merge decide image and write the final blog so to make the artitecture less complex we use subgraph so subgraph is notin but a graph inside a a graph means in the main graph it is used when the system is too complex and also that subgraph hsas it own state so a node can become a subgraph and this project does that but in the project thsis node which is a subgraph does not have its own state but in complex task where 1 node is a agent so this agent has itsown state and he does it work so this way u can tell

ans the major issue in this project is u can tell is evaluation and observabilitu u should've check which node takes how much time and also why much token does each node take what is the latency how would u fix it like this u  can say u would evaluade it by checking tool  using accuracy rhe decision he made and main how much time it take 